# Projet Éthique de l'IA — Adult Income

**Question :** peut-on prédire le revenu annuel tout en limitant les écarts de traitement entre groupes ?

Ce notebook :
- audite les biais associés au sexe et à l'origine déclarée ;
- applique une repondération des exemples ;
- compare fine-tuning complet, LoRA et distillation ;
- mesure performance, équité, temps et taille ;
- explique les décisions avec LIME et SHAP.

> Usage pédagogique uniquement : ce prototype ne doit pas servir à prendre une décision réelle sur une personne.

## 1. Installation

Dans Colab, sélectionner un GPU T4 avant de lancer toutes les cellules.

In [ ]:
!pip -q install "transformers==4.46.3" "datasets==3.1.0" "accelerate==1.1.1" "peft==0.13.2" "shap==0.46.0" "lime==0.2.0.1" "scikit-learn==1.5.2" "seaborn==0.13.2"

In [ ]:
import os, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

warnings.filterwarnings("ignore")
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

BASE_MODEL = "distilbert-base-uncased"
STUDENT_MODEL = "prajjwal1/bert-tiny"
MAX_LENGTH = 128
N_SAMPLES = 3000
RESULTS_DIR = "/content/resultats_projet_ethique"
MODELS_DIR = "/content/modeles_ethique"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device)

## 2. Chargement et préparation des données

Adult Income contient des informations socio-économiques et une cible indiquant si le revenu dépasse 50 000 dollars par an. Les variables sensibles sont conservées pour auditer les écarts, pas pour légitimer leur usage décisionnel.

In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.data.copy()
df["label"] = adult.target.astype(str).str.contains(">50K", regex=False).astype(int).to_numpy()

for col in df.columns:
    if col != "label":
        df[col] = df[col].astype(str).replace({"?": "Unknown", "nan": "Unknown"})

required = ["age", "workclass", "education", "occupation", "hours-per-week", "sex", "race", "label"]
df = df[required].dropna().reset_index(drop=True)
df["stratum"] = df["label"].astype(str) + "_" + df["sex"].astype(str)

sample_n = min(N_SAMPLES, len(df))
df = df.groupby("stratum", group_keys=False).apply(
    lambda x: x.sample(max(1, round(sample_n * len(x) / len(df))), random_state=SEED)
).reset_index(drop=True)
if len(df) > sample_n:
    df = df.sample(sample_n, random_state=SEED).reset_index(drop=True)

print(df.shape)
display(df.head())

## 3. Audit des biais historiques

On mesure le taux de cible positive par groupe. Une différence observée n'est pas automatiquement une preuve de discrimination causale, mais constitue un signal à examiner.

In [ ]:
def positive_rates(frame, sensitive):
    return (
        frame.groupby(sensitive)["label"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "positive_rate"})
        .sort_values("positive_rate", ascending=False)
    )

sex_audit = positive_rates(df, "sex")
race_audit = positive_rates(df, "race")
display(sex_audit)
display(race_audit)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=df, x="sex", y="label", ax=axes[0], errorbar=None)
axes[0].set_title("Taux >50K par sexe")
axes[0].set_ylabel("Taux positif")
sns.barplot(data=df, x="race", y="label", ax=axes[1], errorbar=None)
axes[1].tick_params(axis="x", rotation=35)
axes[1].set_title("Taux >50K par origine déclarée")
axes[1].set_ylabel("Taux positif")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/audit_biais.png", dpi=160)
plt.show()

## 4. Transformation des lignes en texte

Le LLM reçoit une description structurée de chaque personne. Cette approche permet d'utiliser les mêmes architectures que pour une classification de texte.

In [ ]:
FEATURES = ["age", "workclass", "education", "occupation", "hours-per-week", "sex", "race"]

def row_to_text(row):
    return (
        f"Age: {row['age']}. Workclass: {row['workclass']}. "
        f"Education: {row['education']}. Occupation: {row['occupation']}. "
        f"Hours per week: {row['hours-per-week']}. "
        f"Sex: {row['sex']}. Race: {row['race']}."
    )

df["text"] = df.apply(row_to_text, axis=1)
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=SEED, stratify=df["stratum"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(len(train_df), len(test_df))
print(train_df.loc[0, "text"])

## 5. Réduction du biais par repondération

La pondération de Kamiran–Calders rapproche la distribution conjointe sexe/cible du produit des distributions marginales. Les poids sont bornés pour éviter une instabilité excessive.

In [ ]:
def add_reweighing_weights(frame, sensitive="sex"):
    out = frame.copy()
    n = len(out)
    p_s = out[sensitive].value_counts() / n
    p_y = out["label"].value_counts() / n
    p_sy = out.groupby([sensitive, "label"]).size() / n

    def weight(row):
        denom = p_sy.loc[(row[sensitive], row["label"])]
        return (p_s.loc[row[sensitive]] * p_y.loc[row["label"]]) / denom

    out["sample_weight"] = out.apply(weight, axis=1).clip(0.5, 3.0)
    return out

train_fair_df = add_reweighing_weights(train_df)
display(train_fair_df.groupby(["sex", "label"])["sample_weight"].agg(["mean", "count"]))

## 6. Tokenisation et jeux Hugging Face

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_hf(frame, weighted=False):
    payload = {
        "text": frame["text"].tolist(),
        "labels": frame["label"].astype(int).tolist(),
    }
    if weighted:
        payload["sample_weight"] = frame["sample_weight"].astype(float).tolist()
    ds = Dataset.from_dict(payload)
    return ds.map(
        lambda batch: tokenizer(
            batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH
        ),
        batched=True,
        remove_columns=["text"],
    )

train_ds = to_hf(train_df, weighted=False)
train_fair_ds = to_hf(train_fair_df, weighted=True)
test_ds = to_hf(test_df, weighted=False)
test_fair_ds = Dataset.from_dict({
    **{k: test_ds[k] for k in test_ds.column_names},
    "sample_weight": [1.0] * len(test_ds),
})

## 7. Métriques de performance et d'équité

- **Demographic parity gap** : écart maximal des taux de prédiction positive.
- **Equal opportunity gap** : écart maximal des vrais taux positifs.
- Plus ces écarts sont proches de zéro, plus le résultat est homogène entre groupes.

In [ ]:
def classification_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p),
        "recall_macro": float(r),
        "f1_macro": float(f1),
    }

def fairness_gaps(y_true, y_pred, groups):
    tmp = pd.DataFrame({"y": y_true, "pred": y_pred, "group": groups})
    dp = tmp.groupby("group")["pred"].mean()
    eo = tmp[tmp["y"] == 1].groupby("group")["pred"].mean()
    return {
        "demographic_parity_gap": float(dp.max() - dp.min()) if len(dp) > 1 else 0.0,
        "equal_opportunity_gap": float(eo.max() - eo.min()) if len(eo) > 1 else 0.0,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return classification_metrics(labels, preds)

def trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def evaluate_trainer(name, trainer, elapsed, model):
    output = trainer.predict(test_ds)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    pred = np.argmax(logits, axis=-1)
    metrics = classification_metrics(test_df["label"].to_numpy(), pred)
    for sensitive in ["sex", "race"]:
        gaps = fairness_gaps(
            test_df["label"].to_numpy(), pred, test_df[sensitive].to_numpy()
        )
        metrics.update({f"{sensitive}_{k}": v for k, v in gaps.items()})
    trainable, total = trainable_parameters(model)
    metrics.update({
        "model": name,
        "training_seconds": round(elapsed, 1),
        "trainable_parameters": int(trainable),
        "total_parameters": int(total),
    })
    return metrics, pred

## 8. Entraîneur pondéré

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        weights = inputs.pop("sample_weight", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        if weights is None:
            loss = outputs.loss
        else:
            per_item = F.cross_entropy(outputs.logits, labels, reduction="none")
            loss = (per_item * weights.to(per_item.device)).mean()
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = dict(inputs)
        inputs.pop("sample_weight", None)
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)

def args_for(output_dir, epochs=1, lr=2e-5):
    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=25,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        remove_unused_columns=False,
    )

## 9. Méthode A — fine-tuning complet avec repondération

In [ ]:
model_full = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
trainer_full = WeightedTrainer(
    model=model_full,
    args=args_for(f"{MODELS_DIR}/full", epochs=1, lr=2e-5),
    train_dataset=train_fair_ds,
    eval_dataset=test_fair_ds,
    compute_metrics=compute_metrics,
)
start = time.time()
trainer_full.train()
time_full = time.time() - start
metrics_full, pred_full = evaluate_trainer("Full fine-tuning (fair)", trainer_full, time_full, model_full)
model_full.save_pretrained(f"{MODELS_DIR}/full")
metrics_full

## 10. Méthode B — LoRA sans correction

Cette version sert de référence **avant correction**.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

model_lora_raw = get_peft_model(
    AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2),
    lora_config,
)
trainer_lora_raw = Trainer(
    model=model_lora_raw,
    args=args_for(f"{MODELS_DIR}/lora_raw", epochs=1, lr=2e-4),
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
start = time.time()
trainer_lora_raw.train()
time_lora_raw = time.time() - start
metrics_lora_raw, pred_lora_raw = evaluate_trainer(
    "LoRA (original)", trainer_lora_raw, time_lora_raw, model_lora_raw
)
model_lora_raw.save_pretrained(f"{MODELS_DIR}/lora_raw")
metrics_lora_raw

## 11. LoRA après correction

Même architecture et mêmes hyperparamètres, mais entraînement avec les poids d'équité.

In [ ]:
model_lora_fair = get_peft_model(
    AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2),
    lora_config,
)
trainer_lora_fair = WeightedTrainer(
    model=model_lora_fair,
    args=args_for(f"{MODELS_DIR}/lora_fair", epochs=1, lr=2e-4),
    train_dataset=train_fair_ds,
    eval_dataset=test_fair_ds,
    compute_metrics=compute_metrics,
)
start = time.time()
trainer_lora_fair.train()
time_lora_fair = time.time() - start
metrics_lora_fair, pred_lora_fair = evaluate_trainer(
    "LoRA (fair)", trainer_lora_fair, time_lora_fair, model_lora_fair
)
model_lora_fair.save_pretrained(f"{MODELS_DIR}/lora_fair")
metrics_lora_fair

## 12. Méthode C — distillation

Le LoRA corrigé devient le professeur. Un petit BERT apprend à reproduire ses distributions tout en conservant une perte supervisée pondérée.

In [ ]:
class DistillationTrainer(WeightedTrainer):
    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.6, **kwargs):
        super().__init__(**kwargs)
        self.teacher_model = teacher_model.to(device).eval()
        self.temperature = temperature
        self.alpha = alpha
        for p in self.teacher_model.parameters():
            p.requires_grad = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        weights = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher_model(**inputs)

        ce_each = F.cross_entropy(student_outputs.logits, labels, reduction="none")
        if weights is not None:
            ce_loss = (ce_each * weights.to(ce_each.device)).mean()
        else:
            ce_loss = ce_each.mean()

        t = self.temperature
        kd_loss = F.kl_div(
            F.log_softmax(student_outputs.logits / t, dim=-1),
            F.softmax(teacher_outputs.logits / t, dim=-1),
            reduction="batchmean",
        ) * (t ** 2)
        loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, student_outputs) if return_outputs else loss

student_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL, num_labels=2)
trainer_student = DistillationTrainer(
    model=student_model,
    teacher_model=model_lora_fair,
    args=args_for(f"{MODELS_DIR}/student", epochs=2, lr=5e-5),
    train_dataset=train_fair_ds,
    eval_dataset=test_fair_ds,
    compute_metrics=compute_metrics,
)
start = time.time()
trainer_student.train()
time_student = time.time() - start
metrics_student, pred_student = evaluate_trainer(
    "Distillation (fair)", trainer_student, time_student, student_model
)
student_model.save_pretrained(f"{MODELS_DIR}/student")
metrics_student

## 13. Benchmark final

In [ ]:
results = pd.DataFrame([
    metrics_full,
    metrics_lora_raw,
    metrics_lora_fair,
    metrics_student,
]).set_index("model")

display(results.round(4))
results.to_csv(f"{RESULTS_DIR}/benchmark_modeles.csv")
with open(f"{RESULTS_DIR}/benchmark_modeles.json", "w", encoding="utf-8") as f:
    json.dump(results.reset_index().to_dict(orient="records"), f, indent=2, ensure_ascii=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
results["f1_macro"].plot(kind="bar", ax=axes[0], title="F1 macro", ylim=(0, 1))
results["sex_demographic_parity_gap"].plot(
    kind="bar", ax=axes[1], title="Écart de parité — sexe"
)
results["training_seconds"].plot(kind="bar", ax=axes[2], title="Temps d'entraînement (s)")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/benchmark.png", dpi=160)
plt.show()

## 14. Comparaison avant/après correction

In [ ]:
before_after = pd.DataFrame({
    "Avant — LoRA original": metrics_lora_raw,
    "Après — LoRA repondéré": metrics_lora_fair,
}).T[[
    "accuracy", "f1_macro",
    "sex_demographic_parity_gap", "sex_equal_opportunity_gap",
    "race_demographic_parity_gap", "race_equal_opportunity_gap",
]]
display(before_after.round(4))

before_after.to_csv(f"{RESULTS_DIR}/comparaison_avant_apres.csv")

## 15. Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in [
    (axes[0], pred_lora_raw, "LoRA original"),
    (axes[1], pred_lora_fair, "LoRA corrigé"),
]:
    sns.heatmap(
        confusion_matrix(test_df["label"], pred),
        annot=True, fmt="d", cmap="Blues", ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Vérité")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusions_avant_apres.png", dpi=160)
plt.show()

## 16. Fonction de prédiction pour l'XAI

In [ ]:
def predict_proba(texts, model=model_lora_fair):
    model.eval()
    batch = tokenizer(
        list(texts), padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**batch).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

examples = test_df["text"].head(3).tolist()
predict_proba(examples)

## 17. XAI n°1 — LIME

LIME perturbe localement le texte et estime quels mots ont le plus influencé une prédiction particulière.

In [ ]:
from lime.lime_text import LimeTextExplainer

lime_explainer = LimeTextExplainer(class_names=["<=50K", ">50K"])
idx = 0
lime_exp = lime_explainer.explain_instance(
    test_df.loc[idx, "text"],
    predict_proba,
    num_features=10,
    num_samples=500,
)
print("Vraie classe :", test_df.loc[idx, "label"])
print("Probabilités :", predict_proba([test_df.loc[idx, "text"]])[0])
display(pd.DataFrame(lime_exp.as_list(), columns=["élément", "contribution"]))
lime_exp.save_to_file(f"{RESULTS_DIR}/lime_exemple.html")

## 18. XAI n°2 — SHAP

SHAP attribue une contribution à chaque fragment de texte par rapport à une valeur de référence.

In [ ]:
import shap

masker = shap.maskers.Text(tokenizer)
shap_explainer = shap.Explainer(
    lambda texts: predict_proba(list(texts))[:, 1],
    masker,
    output_names=[">50K"],
)
shap_values = shap_explainer(test_df["text"].head(2).tolist())
shap.plots.text(shap_values[0])
shap.plots.bar(shap_values[0], max_display=12)

## 19. Lecture éthique des résultats

Points à commenter après exécution :

1. **Performance** — quel modèle offre le meilleur F1 macro ?
2. **Équité** — la repondération réduit-elle les écarts de parité et d'égalité des chances ?
3. **Compromis** — la baisse éventuelle de performance est-elle proportionnée au gain d'équité ?
4. **Sobriété** — LoRA et la distillation réduisent-ils le nombre de paramètres entraînés ou le temps ?
5. **Explicabilité** — LIME et SHAP font-ils ressortir les variables sensibles ?
6. **Limites** — Adult Income est ancien, américain, incomplet et reflète des inégalités historiques.
7. **Gouvernance** — un humain doit contrôler l'usage, documenter le modèle et prévoir un recours.

Une métrique d'équité plus faible ne suffit pas à rendre le système juste : le choix de la cible, le contexte d'usage et les dommages possibles restent déterminants.

## 20. Trame de présentation — 10 minutes

1. Problème et motivation — 45 s  
2. Dataset et variables sensibles — 1 min  
3. Audit des biais — 1 min  
4. Méthode de repondération — 1 min  
5. Trois méthodes d'adaptation — 1 min 30  
6. Benchmark — 1 min 30  
7. Comparaison avant/après — 1 min  
8. Démonstration LIME + SHAP — 1 min 15  
9. Limites, recommandations et conclusion — 1 min  

Remplacer dans les diapositives tous les résultats provisoires par les valeurs réellement produites dans Colab.